#**CHAPTER 5. TRAINABLE REASONING UNDER GOVERNANCE**
---

##REFERENCE

https://chatgpt.com/share/6999a733-9e50-8012-ae0a-10b172e250df

##0.CONTEXT

**INTRODUCTION (BOARD-FACING) — TRAINABLE REASONING UNDER GOVERNANCE**

You are being asked to evaluate something that is both powerful and easy to misunderstand: how we can use large language models (LLMs) in finance without importing “black-box intuition” into Board-level decision-making. The core point of this notebook is not that an LLM can write a better memo. The core point is that we can build a controlled reasoning pipeline that behaves more like an internal control system than a creative writing engine. In other words: we are not “using AI to decide.” We are using AI inside a governed mechanism that produces auditable artifacts, separates facts from assumptions, and fails closed into human review when controls do not pass.

The specific problem we are solving is familiar to every finance organization: memos get written under time pressure. People synthesize incomplete packets, make implicit assumptions, and present a recommendation. This is normal and often necessary. The risk is that the same workflow, when accelerated by an LLM, can scale not only productivity but also error: unsupported claims can be produced faster, uncertainty can be masked more fluently, and governance can degrade silently because the output “sounds confident.” The Board does not need another tool that generates plausible text. The Board needs a way to ensure that when LLMs are used, the organization can always answer: what inputs were used, what was assumed, what remains unknown, what controls were applied, and what the system did when controls failed.

This notebook is the fifth and most pragmatic step in our Reasoning AI sequence. Earlier notebooks introduced explicit reasoning shapes—chains, trees, loops, committees—each designed to make the inference process legible and bounded. This notebook introduces the principle that makes those architectures operational in production: trainability as measured quality improvement. “Trainable reasoning,” as we define it here, is not fine-tuning mythology. It is a disciplined, testable practice: we change prompts or constraints, we measure what improved or worsened, and we only promote changes that pass regression gates on critical risks. If the change does not pass, we do not argue. We escalate to human review and keep the safer configuration.

The use case is deliberately synthetic and bounded so the mechanism is easy to audit. We feed a synthetic finance packet (income statement, balance sheet highlights, operating KPIs, and a decision question). We instruct the model to produce an investment committee memo in a strict JSON contract with mandatory fields: facts_provided, assumptions_introduced, open_items, questions_to_verify, analysis, draft_output, and verification_status set to “Not verified.” The purpose of this contract is not formatting. The contract is governance: it forces explicit separation of what we know, what we assumed, and what we must verify.

We then run two modes:

First, a baseline prompt. This is the “generic” approach that organizations typically adopt when they first use LLMs: a prompt that asks for a memo and requests the right structure. It may include guardrails, but it is relatively light on enforcement.

Second, an adapted prompt. This is a governed template: a constraints library that explicitly forbids external facts, prohibits market comparables and industry statistics, requires explicit open items and questions, and reinforces the “Not verified” stance. The adapted prompt is not “better writing.” It is a stricter contract intended to reduce the organization’s most material risk: invention and false certainty.

Critically, we do not declare the adapted prompt superior because it feels more disciplined. We test it. We run an evaluation harness with deterministic scoring rules. Deterministic means that the rubric is applied in code, not by the model. This is essential: governance cannot be entirely model-based because that collapses control and subject into the same system. Here, the model generates outputs, and our rubric evaluates them.

The rubric measures exactly the risks that matter in board-facing finance:

Assumption leak rate: a proxy for invented facts. In our implementation, we use a conservative heuristic: we scan the output for numeric claims not present in the packet or in explicitly allowed derived computations (for example, EBITDA derived from packet line items). The point is not perfect detection; the point is a tripwire. If the system starts introducing numbers that do not exist in the packet, we treat that as a control failure and escalate.

Uncertainty disclosure rate: the opposite failure mode. A memo that hides uncertainty is dangerous because it looks complete when it is not. We score uncertainty disclosure by verifying that open_items and questions_to_verify are present and non-empty, and by checking for uncertainty markers in the analysis narrative. Again: this is not about prose; it is about whether the memo forces the reader to see what is missing.

Schema validity: whether the output obeys the JSON contract. This matters because downstream governance and automation rely on structure. If outputs are not structurally valid, controls cannot be applied reliably.

Policy violations: explicit checks for missing “Not verified,” missing open items, missing verification questions, and other contract breaches. The Board should read policy violations as the same class of failure as a broken checklist in a regulated process: not necessarily a catastrophe, but an immediate reason to stop and require review.

We combine these into an overall score with explicit weights. The Board should not over-index on the single overall score; it is a convenience metric. The decision is governed by gates. Gates are hard stops. The evaluation harness produces measurements, but the gates determine whether a configuration can be promoted.

The gating logic is intentionally conservative. We implement regression safety: the adapted prompt must not perform worse on critical metrics. If adapted produces a higher assumption leak rate, more policy violations, or fails schema validity, it is not promoted. It is flagged for human review, and the risk log records why. This is the central institutional lesson: we do not ship improvements on vibes. We ship them only when they measurably reduce risk or at least do not increase it.

The notebook’s outputs are not just the memo. Each run produces a governance bundle—an evidence pack—so an independent reviewer can reconstruct what happened. We write:

A run manifest: configuration, determinism controls, schema hashes, and a run identifier.

A prompts log: hashed and redacted prompt records. We preserve enough to audit prompt changes without storing secrets or sensitive data.

A reasoning trace: this is the “ledger” of what we did. It records the input boundary, the outputs for baseline and adapted modes, the evaluation results, and the control posture. It is not the model’s chain-of-thought; it is an auditable structure describing the pipeline.

A risk log: every control failure is recorded with risk_id, timestamp, severity, category, description, and the control that triggered. This is where governance becomes operational: failures are not hidden; they are logged and escalated.

A final report: board-facing structured JSON that states the recommendation (PROMOTE_ADAPTED or HUMAN_REVIEW), the confidence, and the full reasoning_quality_report. Verification status is always “Not verified” to prevent accidental upgrading of synthetic output into presumed fact.

Deliverables are packaged into a zip for distribution and archiving.

Why is this relevant to the Board? Because it converts LLM usage from an informal drafting habit into a controlled institutional workflow. It creates a repeatable method for evaluating prompt changes, enforcing contracts, and preventing “silent regressions” where outputs become less safe over time. In the same way that we would not change a credit policy without measuring impacts, we should not change an LLM prompt template used in finance without a regression harness.

This notebook also establishes a governance posture that scales. Today we evaluate baseline vs adapted prompts. Tomorrow we can evaluate different constraint libraries for different business lines: advisory memos, credit underwriting summaries, risk committee dashboards, or suitability notes. The mechanism stays the same: bounded inputs, explicit contracts, deterministic evaluation, and hard stop gates.

The Board should interpret the results in practical terms. If the notebook recommends PROMOTE_ADAPTED, it means: under this rubric, the stricter template produced at least as safe outputs as the baseline, and it did not regress on critical risk measures. It does not mean the memo is “true.” It means the pipeline behaved as designed and is safer to operationalize in drafting contexts with human review.

If the notebook recommends HUMAN_REVIEW, it means: the adapted prompt did not meet regression safety thresholds, or schema/policy checks failed. This is a safe outcome. A control system that never triggers review is a control system that is not checking anything meaningful.

Finally, we must be explicit about the boundary of what we are claiming. This notebook does not verify external truth. It does not pull market data. It does not validate financial statements. It does not replace professional judgment. Its purpose is to build the “safety rails” for reasoning: structure, measurement, evidence, and escalation. That is precisely what the Board should demand before permitting broader adoption.

In short: what we are doing here is building a finance-grade reasoning pipeline that turns LLM drafting into an auditable, measurable, governed process. The relevance is institutional: higher throughput without surrendering control, and a repeatable way to improve templates safely over time.

##1.LIBRARIES AND ENVIRONMENT

**CELL 1 — INSTALL, IMPORTS, DETERMINISM, DIRECTORY SETUP**

Cell 1 establishes the execution environment so that everything that follows is repeatable, inspectable, and portable across Colab runs. We install only the minimal dependencies required for this notebook: the Anthropic client library (to call Claude) and jsonschema (to validate output contracts). The design goal is “production-clean”: the fewer moving parts, the fewer ungoverned failure modes.

Next, the cell imports standard libraries used across the pipeline: hashing (for evidence and integrity), random (for deterministic synthetic data), datetime with timezone awareness (for UTC timestamps), zipfile (for packaging deliverables), and typing utilities (to make the code easier to audit). This notebook’s posture is not “prototype convenience.” It is “reviewability.” Strong typing and stable imports reduce ambiguity and reduce the chance that future modifications change behavior silently.

Determinism controls begin here. We set PYTHONHASHSEED to a fixed value and fix random.seed. PYTHONHASHSEED matters because Python hash randomization can change ordering behavior in subtle ways, which can alter JSON output ordering or derived computations. A fixed seed ensures stable behavior for deterministic synthetic case generation and any scoring heuristics that rely on consistent iteration order.

We then create the artifacts/ and deliverables/ directories. This is the governance boundary: every run writes a complete evidence bundle into artifacts and packages a zip into deliverables. In institutional settings, artifacts are not optional. Without them, results are not auditable. By creating the directories at the start, we avoid silent failures later where writes fail due to missing paths.

Finally, Cell 1 defines a UTC timestamp helper that uses timezone-aware datetime.now(datetime.timezone.utc). This is a compliance-grade choice. It avoids ambiguous local times and prevents errors where logs cannot be sequenced reliably across systems. Every subsequent artifact uses this timestamp function to standardize event timing across manifest, trace, risk log, and report.

In short, Cell 1 creates the “controlled lab bench” for the notebook: minimal dependencies, deterministic execution, standardized logging time, and a stable file structure for evidence production.

In [1]:
# Cell 1 — Install + imports + deterministic settings + directory setup
!pip -q install "anthropic>=0.49.0" "jsonschema>=4.22.0"

import os, json, re, hashlib, random, zipfile, textwrap
import datetime
from typing import Any, Dict, List, Tuple, Optional

from jsonschema import validate as jsonschema_validate, Draft202012Validator
from google.colab import userdata

os.environ["PYTHONHASHSEED"] = "1337"
random.seed(1337)

BASE_DIR = os.getcwd()
ARTIFACTS_DIR = os.path.join(BASE_DIR, "artifacts")
DELIVERABLES_DIR = os.path.join(BASE_DIR, "deliverables")
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
os.makedirs(DELIVERABLES_DIR, exist_ok=True)

def now_utc_iso() -> str:
    return datetime.datetime.now(datetime.timezone.utc).isoformat()

print("OK:", {"artifacts": ARTIFACTS_DIR, "deliverables": DELIVERABLES_DIR, "ts_utc": now_utc_iso()})

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.0/457.0 kB 16.7 MB/s eta 0:00:00
OK: {'artifacts': '/content/artifacts', 'deliverables': '/content/deliverables', 'ts_utc': '2026-02-21T12:21:37.514417+00:00'}


##2.CONFIGURATION AND SCHEMA

###2.1.OVERVIEW

**CELL 2 — CONFIGURATION, SCHEMAS, AND GOVERNANCE HELPERS**

Cell 2 defines the notebook’s explicit governance policy. First, it creates a configuration object that includes run identifiers, model choice, token limits, and scoring weights. By putting these in config rather than scattered constants, we make the system change-managed: reviewers can see exactly which policy settings were in effect for a run, and future changes can be tracked.

Next, Cell 2 defines strict JSON schemas for all outputs: the IC memo schema, the reasoning quality report schema, and the final report schema. This is not a formatting preference. It is a control mechanism. Schema validation ensures that outputs contain the required fields (facts, assumptions, open items, questions, analysis, draft output, verification status) and do not contain unexpected fields that might hide unreviewed content. In governance terms, schemas establish the “contract” between model output and downstream review processes.

Then we implement shared helpers required by the governance bundle: write_json and append_jsonl for consistent artifact writing; sha256_text and sha256_json for integrity hashing; and redaction logic to prevent secrets or PII-like strings from being written into logs. Prompt logs are particularly sensitive because they can accidentally capture keys or personal identifiers. Even in a synthetic notebook, we treat redaction as mandatory so the pattern is production-ready.

Validation is implemented as a reusable function that returns an error string rather than raising. This is important because the system must fail closed but still log what failed. If validation throws and stops the notebook, we lose audit evidence. Instead, we capture the failure, log it as a risk, and route the decision to HUMAN_REVIEW.

Cell 2 therefore establishes two foundational pillars: explicit policy parameters and enforceable structural contracts. A Board should recognize this as the difference between “we asked an AI to do something” and “we built an auditable system with clear constraints.” This is the cell where governance becomes code: explicit requirements, explicit schemas, explicit integrity hashing, and explicit redaction boundaries.

###2.2.CODE AND IMPLEMENTATION

In [2]:
# Cell 2 — Config + schemas + helpers (hashing, redaction, JSON writing, validation)
CONFIG: Dict[str, Any] = {
    "run_id": hashlib.sha256((now_utc_iso() + "|trainable_reasoning").encode("utf-8")).hexdigest()[:16],
    "model": "claude-haiku-4-5-20251001",
    "max_tokens": 1200,
    "temperature": 0.0,
    "rubric_weights": {
        "schema_validity": 0.35,
        "policy_violations": 0.25,
        "assumption_leak_rate": 0.25,
        "uncertainty_disclosure_rate": 0.15,
    },
    "critical_metrics": ["schema_validity", "policy_violations", "assumption_leak_rate"],
    "regression_thresholds": {
        "assumption_leak_rate_max_increase": 0.0,   # adapted must be <= baseline
        "policy_violations_max_increase": 0,        # adapted must be <= baseline
        "schema_validity_required": True,           # adapted must pass schema
    },
    "paths": {
        "run_manifest": os.path.join(ARTIFACTS_DIR, "run_manifest.json"),
        "prompts_log": os.path.join(ARTIFACTS_DIR, "prompts_log.jsonl"),
        "reasoning_trace": os.path.join(ARTIFACTS_DIR, "reasoning_trace.json"),
        "risk_log": os.path.join(ARTIFACTS_DIR, "risk_log.json"),
        "final_report": os.path.join(ARTIFACTS_DIR, "final_report.json"),
        "deliverables_zip": os.path.join(DELIVERABLES_DIR, "deliverables.zip"),
    },
    "termination": {
        "max_llm_calls": 2,
    },
}

IC_MEMO_SCHEMA: Dict[str, Any] = {
    "type": "object",
    "additionalProperties": False,
    "required": [
        "executive_summary",
        "facts_provided",
        "assumptions_introduced",
        "open_items",
        "questions_to_verify",
        "analysis",
        "draft_output",
        "verification_status",
    ],
    "properties": {
        "executive_summary": {"type": "string", "minLength": 1},
        "facts_provided": {"type": "object"},
        "assumptions_introduced": {"type": "array", "items": {"type": "string"}},
        "open_items": {"type": "array", "items": {"type": "string"}},
        "questions_to_verify": {"type": "array", "items": {"type": "string"}},
        "analysis": {"type": "string", "minLength": 1},
        "draft_output": {"type": "string", "minLength": 1},
        "verification_status": {"type": "string", "enum": ["Not verified"]},
    },
}

QUALITY_REPORT_SCHEMA: Dict[str, Any] = {
    "type": "object",
    "additionalProperties": False,
    "required": [
        "baseline_output",
        "adapted_output",
        "metrics_comparison",
        "failure_examples",
        "deployment_constraints_recommendation",
        "verification_status",
    ],
    "properties": {
        "baseline_output": IC_MEMO_SCHEMA,
        "adapted_output": IC_MEMO_SCHEMA,
        "metrics_comparison": {"type": "object"},
        "failure_examples": {"type": "array", "items": {"type": "object"}},
        "deployment_constraints_recommendation": {"type": "string", "minLength": 1},
        "verification_status": {"type": "string", "enum": ["Not verified"]},
    },
}

FINAL_REPORT_SCHEMA: Dict[str, Any] = {
    "type": "object",
    "additionalProperties": False,
    "required": [
        "run_id",
        "timestamp_utc",
        "facts_provided",
        "assumptions_introduced",
        "open_items",
        "questions_to_verify",
        "analysis",
        "draft_output",
        "verification_status",
        "reasoning_quality_report",
        "recommendation",
        "confidence",
    ],
    "properties": {
        "run_id": {"type": "string"},
        "timestamp_utc": {"type": "string"},
        "facts_provided": {"type": "object"},
        "assumptions_introduced": {"type": "array", "items": {"type": "string"}},
        "open_items": {"type": "array", "items": {"type": "string"}},
        "questions_to_verify": {"type": "array", "items": {"type": "string"}},
        "analysis": {"type": "string"},
        "draft_output": {"type": "string"},
        "verification_status": {"type": "string", "enum": ["Not verified"]},
        "reasoning_quality_report": QUALITY_REPORT_SCHEMA,
        "recommendation": {"type": "string", "enum": ["PROMOTE_ADAPTED", "HUMAN_REVIEW"]},
        "confidence": {"type": "string", "enum": ["low", "medium", "high"]},
    },
}

def sha256_text(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

def sha256_json(obj: Any) -> str:
    return sha256_text(json.dumps(obj, sort_keys=True, separators=(",", ":")))

def write_json(path: str, obj: Any) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, sort_keys=True)

def append_jsonl(path: str, obj: Any) -> None:
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False, sort_keys=True) + "\n")

PII_PATTERNS = [
    (re.compile(r"\b\d{3}-\d{2}-\d{4}\b"), "[REDACTED_SSN]"),
    (re.compile(r"\b(?:\+?\d{1,3}[\s-]?)?(?:\(?\d{2,3}\)?[\s-]?)?\d{3}[\s-]?\d{4}\b"), "[REDACTED_PHONE]"),
    (re.compile(r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b", re.IGNORECASE), "[REDACTED_EMAIL]"),
]

def redact_text(s: str) -> str:
    red = s
    for pat, repl in PII_PATTERNS:
        red = pat.sub(repl, red)
    red = red.replace(str(userdata.get("ANTHROPIC_API_KEY") or ""), "[REDACTED_KEY]")
    return red

def validate_or_none(schema: Dict[str, Any], obj: Any) -> Optional[str]:
    try:
        jsonschema_validate(instance=obj, schema=schema)
        return None
    except Exception as e:
        return str(e)

print("OK:", {"run_id": CONFIG["run_id"], "schema_hash": sha256_json(IC_MEMO_SCHEMA)[:12]})

OK: {'run_id': '08f03fcfd0676eab', 'schema_hash': 'c45dfb066103'}


##3.SYNTHETIC FINNCE CASE GENERATOR

###3.1.OVERVIEW

**CELL 3 — DETERMINISTIC SYNTHETIC CASE AND INPUT BOUNDARY**

Cell 3 produces the synthetic finance packet that serves as the only allowed input to the model. We keep the packet deterministic: fixed values for revenue, costs, leverage proxies, and operating KPIs. This matters because the notebook is an evaluation harness; if the input changes, results cannot be compared run-to-run. A stable synthetic case isolates the effect of prompt changes, which is the subject of this notebook.

The packet includes three components that are governance-relevant. First, a bounded facts section: an income statement, balance sheet highlights, and KPIs. Second, an explicit list of known missing items. This is deliberate. We want the system to demonstrate uncertainty disclosure and diligence discipline: it must surface missing items as open questions rather than “fill in the blanks.” Third, an input boundary object specifying allowed and disallowed sources. This boundary becomes part of the reasoning trace so the run can be audited. It is a policy declaration embedded in the artifact, not an informal instruction.

Cell 3 also computes a small set of explicitly allowed derived values (for example EBITDA and leverage proxy). This is important because finance memos must contain calculations, and we should not treat all derived numbers as hallucinations. The notebook therefore distinguishes between “derived from provided facts” and “invented facts.” We declare the allowed derivations in code so reviewers can validate them.

These allowed derived values are then used by the assumption leak heuristic later: if the output introduces numbers not present in the packet and not in the allowed derived set, we treat them as potential invented facts. This is conservative and intentionally simple. Its role is to detect the most common high-risk failure quickly and deterministically.

Overall, Cell 3 defines the “truth boundary” for the run: what is provided, what is missing, what may be derived, and what must never be imported. This is the foundation on which prompt governance and evaluation credibility rest.

###3.2.CODE AND IMPLEMENTATION

In [6]:
# Cell 3 — Synthetic finance case generator (deterministic) + input boundary definition
def synthetic_packet(seed: int = 1337) -> Dict[str, Any]:
    rnd = random.Random(seed)
    revenue = 240.0  # $m
    cogs = 150.0
    opex = 62.0
    da = 10.0
    interest = 8.0
    tax = 2.0

    total_debt = 95.0
    cash = 18.0
    ar = 34.0
    ap = 22.0
    inventory = 28.0

    # Deterministic “qualitative facts”
    segment = "specialty components"
    customer_concentration_top3 = 0.46
    backlog_months = 5
    capacity_utilization = 0.78

    packet = {
        "company_name": "Northbridge Components (Synthetic)",
        "period": "LTM",
        "currency": "USD (synthetic)",
        "income_statement_usd_m": {
            "revenue": revenue,
            "cogs": cogs,
            "opex": opex,
            "depreciation_amortization": da,
            "interest_expense": interest,
            "tax_expense": tax,
        },
        "balance_sheet_highlights_usd_m": {
            "total_debt": total_debt,
            "cash": cash,
            "accounts_receivable": ar,
            "accounts_payable": ap,
            "inventory": inventory,
        },
        "operating_kpis": {
            "segment": segment,
            "customer_concentration_top3": customer_concentration_top3,
            "backlog_months": backlog_months,
            "capacity_utilization": capacity_utilization,
        },
        "decision_question": "Draft a short IC memo from this packet. Do not add any external facts. Separate facts/assumptions/open items. Board-ready, control-grade.",
        "known_missing_items": [
            "Capex history and maintenance vs growth split",
            "Working capital seasonality detail",
            "Customer contract terms and renewal risk",
            "Recent pricing actions and pass-through constraints",
            "Near-term debt maturities and covenants",
        ],
        "input_boundary": {
            "allowed_sources": ["this_packet_only"],
            "disallowed": ["market comps", "industry growth rates", "macroeconomic forecasts", "unnamed sources"],
        },
    }
    return packet

CASE = synthetic_packet(seed=1337)

# Deterministic derived numbers we will allow (computed from packet facts)
def compute_allowed_derived(case: Dict[str, Any]) -> Dict[str, float]:
    is_ = case["income_statement_usd_m"]
    revenue = float(is_["revenue"])
    cogs = float(is_["cogs"])
    opex = float(is_["opex"])
    da = float(is_["depreciation_amortization"])
    ebitda = revenue - cogs - opex + da
    ebitda_margin = ebitda / revenue if revenue else 0.0
    net_debt = float(case["balance_sheet_highlights_usd_m"]["total_debt"]) - float(case["balance_sheet_highlights_usd_m"]["cash"])
    leverage_proxy = net_debt / ebitda if ebitda else 0.0
    return {
        "ebitda_usd_m": round(ebitda, 4),
        "ebitda_margin": round(ebitda_margin, 6),
        "net_debt_usd_m": round(net_debt, 4),
        "net_debt_to_ebitda": round(leverage_proxy, 6),
    }

ALLOWED_DERIVED = compute_allowed_derived(CASE)

print("OK:", {"company": CASE["company_name"], "allowed_derived": ALLOWED_DERIVED})

OK: {'company': 'Northbridge Components (Synthetic)', 'allowed_derived': {'ebitda_usd_m': 38.0, 'ebitda_margin': 0.158333, 'net_debt_usd_m': 77.0, 'net_debt_to_ebitda': 2.026316}}


##4.LLM CLIEN WRAPPER

###4.1.OVERVIEW

**CELL 4 — LLM CLIENT WRAPPER AND PROMPT LOGGING**

Cell 4 is the controlled interface between our governed pipeline and the external model. We retrieve the Anthropic API key from Colab Secrets, not from plain text. This is operational hygiene and also a pattern the Board should insist on in production: secrets must be managed by platform controls, not embedded in notebooks.

The client wrapper enforces three governance properties. First, we centralize the model choice (claude-haiku-4-5-20251001) and generation parameters. Second, we log every prompt invocation in a redacted, hashed form. We store prompt hashes and a small redacted preview so we can audit changes without capturing sensitive details. This supports a “prompt change ledger” similar to a policy change record: we can prove which prompt was used without exposing it broadly.

Third, we implement strict JSON parsing. The wrapper expects that the model returns valid JSON only. If the model returns markdown-wrapped JSON, we strip the fences deterministically. If parsing fails, the wrapper returns None and the pipeline fails closed: it logs a high-severity risk and replaces the output with a safe HUMAN_REVIEW object. This behavior is critical. It ensures that a formatting mistake does not silently flow into downstream decisions.

We also define a system instruction that tells the model it must not introduce external facts and must always set verification_status to “Not verified.” This is a repeated control: policy is enforced both by instruction and by downstream gates. In governance terms, we use defense-in-depth. If the model follows instructions, great. If it does not, the gates catch violations.

Cell 4 therefore turns model interaction into an auditable, bounded API call: logged, redacted, and structurally constrained. This is the minimum required posture for using LLMs in finance workflows.

###4.2.CODE AND IMPLEMENTATION

In [5]:
# Cell 4 — LLM client wrapper (Anthropic) + prompt logging (redacted + hashes)
from anthropic import Anthropic

API_KEY = userdata.get("ANTHROPIC_API_KEY")
if not API_KEY:
    raise RuntimeError("Missing Colab Secret ANTHROPIC_API_KEY")

client = Anthropic(api_key=API_KEY)

def _extract_text(resp: Any) -> str:
    # anthropic SDK returns content blocks
    try:
        blocks = resp.content
        if isinstance(blocks, list) and blocks and hasattr(blocks[0], "text"):
            return "".join([b.text for b in blocks if hasattr(b, "text")])
    except Exception:
        pass
    # fallback
    return str(resp)

def llm_json_call(prompt_name: str, system: str, user: str, max_tokens: int, temperature: float) -> Tuple[Optional[Dict[str, Any]], str]:
    ts = now_utc_iso()
    raw_user = user
    red_user = redact_text(raw_user)
    entry = {
        "timestamp_utc": ts,
        "run_id": CONFIG["run_id"],
        "prompt_name": prompt_name,
        "model": CONFIG["model"],
        "system_sha256": sha256_text(system),
        "user_redacted_sha256": sha256_text(red_user),
        "user_redacted_preview": red_user[:500],
    }
    append_jsonl(CONFIG["paths"]["prompts_log"], entry)

    resp = client.messages.create(
        model=CONFIG["model"],
        max_tokens=max_tokens,
        temperature=temperature,
        system=system,
        messages=[{"role": "user", "content": raw_user}],
    )
    text = _extract_text(resp).strip()

    # Strict JSON parse; if wrapped in ```json ... ``` strip it deterministically
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z]*\s*", "", text)
        text = re.sub(r"\s*```$", "", text).strip()

    try:
        obj = json.loads(text)
        return obj, text
    except Exception:
        return None, text

SYSTEM_JSON_ONLY = (
    "You are a governance-first finance assistant. Output MUST be valid JSON only (no markdown, no prose outside JSON). "
    "Use ONLY facts from the provided packet. Do NOT add external facts, market data, or sources. "
    "If something is missing, put it in open_items and questions_to_verify. "
    "Always set verification_status to 'Not verified'."
)

print("OK:", {"model": CONFIG["model"], "max_calls": CONFIG["termination"]["max_llm_calls"]})

OK: {'model': 'claude-haiku-4-5-20251001', 'max_calls': 2}


##5.EVALUATION HARNESS

###5.1.OVERVIEW

**CELL 5 — EVALUATION HARNESS: SCORING AND COMPARISON**

Cell 5 is the heart of “trainable reasoning” as measured improvement. It implements deterministic evaluation functions that score model outputs without relying on the model to judge itself. This is the control layer. It makes quality measurable and therefore governable.

We define a numeric extraction helper that identifies numbers in text. This supports the assumption leak heuristic: the output memo should not contain numeric facts not present in the input packet or in explicitly allowed derivations. We build a set of permitted numbers by scanning the packet JSON and adding allowed derived values. Then we scan the generated memo’s executive summary, analysis, draft output, and assumptions. Numbers not in the permitted set are treated as leaks. The leak rate is computed as leaked numbers divided by total numbers found. This is a conservative approach: it may flag benign numbers, but it is a safe tripwire for hallucinated quantitative claims.

We also score uncertainty disclosure. The notebook requires open_items and questions_to_verify. The evaluation checks that they exist and are non-empty. It also looks for uncertainty markers in the analysis text. The intention is not stylistic policing; it is ensuring that outputs preserve uncertainty rather than compress it away.

Schema validity and policy violations are recorded as structured events. Missing “Not verified,” missing open items, missing questions, and schema failures are counted. These are the control failures that most directly correspond to governance breakdown in real usage.

Finally, we compute an overall score using explicit weights in config. The Board should interpret this as an index, not a decision. The decision comes from gates. The role of the overall score is to provide a summary for trend tracking across prompt iterations.

The compare function produces a metrics_comparison object and standardized failure examples. This becomes part of the reasoning_quality_report and the reasoning trace. Cell 5 thus creates the evaluative “measurement harness” that makes prompt adaptation safe, testable, and change-managed.

###5.2.CODE AND IMPLEMENTATION

In [7]:
# Cell 5 — Evaluation harness (REQUIRED): score_output + compare
def _collect_numbers(text: str) -> List[str]:
    # captures integers/decimals/percents; normalize by stripping trailing % and commas
    raw = re.findall(r"(?<![A-Za-z])[-+]?\d{1,3}(?:,\d{3})*(?:\.\d+)?%?|[-+]?\d+(?:\.\d+)?%?", text)
    out = []
    for x in raw:
        x2 = x.replace(",", "")
        if x2.endswith("%"):
            x2 = x2[:-1]
        out.append(x2)
    return out

def _facts_number_set(case: Dict[str, Any], allowed_derived: Dict[str, float]) -> set:
    blob = json.dumps(case, sort_keys=True)
    nums = set(_collect_numbers(blob))
    for v in allowed_derived.values():
        nums.add(str(v))
        # also add common rounded variants
        nums.add(str(round(float(v), 2)))
        nums.add(str(round(float(v), 3)))
    return nums

FACT_NUMS = _facts_number_set(CASE, ALLOWED_DERIVED)

def _uncertainty_markers(text: str) -> int:
    markers = ["uncertain", "unknown", "requires", "needs verification", "to be confirmed", "open item", "cannot verify"]
    t = text.lower()
    return sum(1 for m in markers if m in t)

def score_output(output_obj: Any, facts_case: Dict[str, Any]) -> Dict[str, Any]:
    metrics: Dict[str, Any] = {
        "schema_validity": False,
        "assumption_leak_rate": 1.0,
        "uncertainty_disclosure_rate": 0.0,
        "policy_violations": [],
        "overall_score": 0.0,
        "debug": {},
    }

    # Schema validity
    err = validate_or_none(IC_MEMO_SCHEMA, output_obj)
    if err is None:
        metrics["schema_validity"] = True
    else:
        metrics["policy_violations"].append({"type": "SCHEMA_INVALID", "detail": err})

    # Policy checks (even if schema invalid, be robust)
    vs = (output_obj.get("verification_status") if isinstance(output_obj, dict) else None)
    if vs != "Not verified":
        metrics["policy_violations"].append({"type": "MISSING_NOT_VERIFIED", "detail": f"verification_status={vs!r}"})

    open_items = output_obj.get("open_items") if isinstance(output_obj, dict) else None
    questions = output_obj.get("questions_to_verify") if isinstance(output_obj, dict) else None
    if not isinstance(open_items, list) or len(open_items) == 0:
        metrics["policy_violations"].append({"type": "MISSING_OPEN_ITEMS", "detail": "open_items must be non-empty list"})
    if not isinstance(questions, list) or len(questions) == 0:
        metrics["policy_violations"].append({"type": "MISSING_QUESTIONS", "detail": "questions_to_verify must be non-empty list"})

    # Assumption leak heuristic: new numbers not in facts + allowed derived
    text_parts = []
    if isinstance(output_obj, dict):
        for k in ["executive_summary", "analysis", "draft_output"]:
            v = output_obj.get(k, "")
            if isinstance(v, str):
                text_parts.append(v)
        if isinstance(output_obj.get("assumptions_introduced"), list):
            text_parts.extend([x for x in output_obj["assumptions_introduced"] if isinstance(x, str)])

    joined = "\n".join(text_parts)
    out_nums = _collect_numbers(joined)
    leaks = [n for n in out_nums if n not in FACT_NUMS]

    total_nums = max(1, len(out_nums))
    leak_rate = len(leaks) / total_nums
    metrics["assumption_leak_rate"] = float(round(leak_rate, 6))
    if len(leaks) > 0:
        metrics["policy_violations"].append({"type": "POTENTIAL_INVENTED_NUMERIC_FACTS", "detail": {"leaks": leaks[:25], "count": len(leaks)}})

    # Uncertainty disclosure (deterministic): open_items + questions + uncertainty markers in analysis
    has_open = isinstance(open_items, list) and len(open_items) > 0
    has_q = isinstance(questions, list) and len(questions) > 0
    markers = _uncertainty_markers(str(output_obj.get("analysis", "")) if isinstance(output_obj, dict) else "")
    u = 0.0
    u += 0.5 if has_open else 0.0
    u += 0.5 if has_q else 0.0
    u = u + 0.1 if markers > 0 else u
    metrics["uncertainty_disclosure_rate"] = float(min(1.0, round(u, 6)))

    # Overall score (higher is better): schema_validity, fewer violations, lower leak, higher uncertainty disclosure
    w = CONFIG["rubric_weights"]
    schema_score = 1.0 if metrics["schema_validity"] else 0.0
    violation_count = len(metrics["policy_violations"])
    violations_score = max(0.0, 1.0 - (violation_count / 6.0))  # cap at 6+
    leak_score = max(0.0, 1.0 - metrics["assumption_leak_rate"])
    uncertainty_score = metrics["uncertainty_disclosure_rate"]

    overall = (
        w["schema_validity"] * schema_score +
        w["policy_violations"] * violations_score +
        w["assumption_leak_rate"] * leak_score +
        w["uncertainty_disclosure_rate"] * uncertainty_score
    )
    metrics["overall_score"] = float(round(overall, 6))
    metrics["debug"] = {
        "out_nums_count": len(out_nums),
        "leaks_count": len(leaks),
        "violation_count": violation_count,
        "uncertainty_markers": markers,
    }
    return metrics

def compare(baseline_obj: Dict[str, Any], adapted_obj: Dict[str, Any], facts_case: Dict[str, Any]) -> Dict[str, Any]:
    baseline_metrics = score_output(baseline_obj, facts_case)
    adapted_metrics = score_output(adapted_obj, facts_case)

    # Comparison deltas
    comp = {
        "baseline": baseline_metrics,
        "adapted": adapted_metrics,
        "deltas": {
            "assumption_leak_rate": float(round(adapted_metrics["assumption_leak_rate"] - baseline_metrics["assumption_leak_rate"], 6)),
            "overall_score": float(round(adapted_metrics["overall_score"] - baseline_metrics["overall_score"], 6)),
            "policy_violations": len(adapted_metrics["policy_violations"]) - len(baseline_metrics["policy_violations"]),
            "schema_validity": (adapted_metrics["schema_validity"] and not baseline_metrics["schema_validity"]) or (baseline_metrics["schema_validity"] and not adapted_metrics["schema_validity"]),
        },
    }

    # Failure examples: show top violation types
    def _summarize_violations(metrics: Dict[str, Any]) -> Dict[str, int]:
        counts: Dict[str, int] = {}
        for v in metrics.get("policy_violations", []):
            t = v.get("type", "UNKNOWN")
            counts[t] = counts.get(t, 0) + 1
        return counts

    failure_examples = [
        {"mode": "baseline", "violation_counts": _summarize_violations(baseline_metrics)},
        {"mode": "adapted", "violation_counts": _summarize_violations(adapted_metrics)},
    ]

    # Deployment constraints recommendation (deterministic template)
    rec = []
    rec.append("Keep strict JSON-only output and schema validation as a hard gate.")
    rec.append("Maintain explicit facts/assumptions/open-items separation; never allow external data injection.")
    rec.append("Keep numeric-leak heuristic and expand with stronger grounding checks in production.")
    rec.append("Require human review on any regression in critical metrics or any schema/policy failure.")
    deployment_constraints = " ".join(rec)

    report = {
        "baseline_output": baseline_obj,
        "adapted_output": adapted_obj,
        "metrics_comparison": comp,
        "failure_examples": failure_examples,
        "deployment_constraints_recommendation": deployment_constraints,
        "verification_status": "Not verified",
    }
    return report

print("OK:", {"facts_nums_count": len(FACT_NUMS), "weights": CONFIG["rubric_weights"]})

OK: {'facts_nums_count': 23, 'weights': {'schema_validity': 0.35, 'policy_violations': 0.25, 'assumption_leak_rate': 0.25, 'uncertainty_disclosure_rate': 0.15}}


##6.GATES AND RISK DETECTION

###6.1.OVERVIEW

**CELL 6 — GATES, RISK DETECTION, AND ESCALATION**

Cell 6 converts measurements into governance outcomes. It defines the risk log structure and the hard stop rules. The risk log is not just a list of issues; it is a structured register with risk_id, timestamp, severity, category, description, control, and status. This mirrors how operational risk functions in financial institutions: issues must be categorized, controlled, and escalated.

We then implement three classes of gates.

Schema gate: if the model output violates the JSON schema, we log a high-severity risk and treat the output as requiring HUMAN_REVIEW. This ensures that downstream systems do not operate on malformed data.

Policy gate: even if schema passes, policy can still fail. For example, the model might omit open items or fail to state “Not verified.” These are governance-critical omissions. The policy gate logs medium to high risks and routes the run to HUMAN_REVIEW.

Regression safety gate: this is the defining concept for trainable reasoning. We do not promote the adapted prompt if it performs worse on critical metrics than the baseline. Critical metrics include schema validity, policy violations, and assumption leakage. The thresholds are explicit: the adapted configuration must not increase leaks or policy violations and must pass schema. If it fails, we log a high-severity regression risk and set decision to HUMAN_REVIEW.

The logic is intentionally strict because we are building a pattern that must survive real-world incentives. In practice, teams will be tempted to promote a prompt because it “reads better.” Regression gates prevent that. They force promotion decisions to be justified by measured safety and contract compliance.

Cell 6 is where the notebook becomes institution-ready: it encodes the principle that improvements are only improvements when they pass controls. Everything else is experimentation that requires human oversight.

###6.2.CODE AND IMPLEMENTATION

In [8]:
# Cell 6 — Gates + risk detection + escalation logic
RISK_LOG: List[Dict[str, Any]] = []

def log_risk(severity: str, category: str, description: str, control: str, status: str) -> None:
    rid = hashlib.sha256((CONFIG["run_id"] + "|" + now_utc_iso() + "|" + category + "|" + description).encode("utf-8")).hexdigest()[:12]
    RISK_LOG.append({
        "risk_id": rid,
        "timestamp_utc": now_utc_iso(),
        "severity": severity,
        "category": category,
        "description": description,
        "control": control,
        "status": status,
    })

def gate_schema(output_obj: Any, mode: str) -> bool:
    err = validate_or_none(IC_MEMO_SCHEMA, output_obj)
    if err is not None:
        log_risk("high", "SCHEMA", f"{mode}: IC memo schema invalid: {err}", "JSON schema validation", "HUMAN_REVIEW")
        return False
    return True

def gate_policy(output_obj: Any, mode: str) -> bool:
    ok = True
    if not isinstance(output_obj, dict):
        log_risk("high", "POLICY", f"{mode}: output is not a JSON object", "Strict JSON object requirement", "HUMAN_REVIEW")
        return False

    if output_obj.get("verification_status") != "Not verified":
        ok = False
        log_risk("high", "POLICY", f"{mode}: verification_status must be 'Not verified'", "Mandatory disclaimer", "HUMAN_REVIEW")

    if not isinstance(output_obj.get("open_items"), list) or len(output_obj["open_items"]) == 0:
        ok = False
        log_risk("medium", "POLICY", f"{mode}: open_items missing/empty", "Uncertainty disclosure requirement", "HUMAN_REVIEW")

    if not isinstance(output_obj.get("questions_to_verify"), list) or len(output_obj["questions_to_verify"]) == 0:
        ok = False
        log_risk("medium", "POLICY", f"{mode}: questions_to_verify missing/empty", "Verification questions requirement", "HUMAN_REVIEW")

    return ok

def gate_regression_safety(baseline_metrics: Dict[str, Any], adapted_metrics: Dict[str, Any]) -> bool:
    ok = True
    thr = CONFIG["regression_thresholds"]

    if thr["schema_validity_required"] and not adapted_metrics.get("schema_validity", False):
        ok = False
        log_risk("high", "REGRESSION", "Adapted schema_validity failed (required pass).", "Gate A: Regression safety", "HUMAN_REVIEW")

    delta_leak = adapted_metrics.get("assumption_leak_rate", 1.0) - baseline_metrics.get("assumption_leak_rate", 1.0)
    if delta_leak > thr["assumption_leak_rate_max_increase"]:
        ok = False
        log_risk("high", "REGRESSION", f"Adapted assumption_leak_rate worse by {delta_leak:.6f}.", "Gate A: Regression safety", "HUMAN_REVIEW")

    delta_viol = len(adapted_metrics.get("policy_violations", [])) - len(baseline_metrics.get("policy_violations", []))
    if delta_viol > thr["policy_violations_max_increase"]:
        ok = False
        log_risk("high", "REGRESSION", f"Adapted policy_violations increased by {delta_viol}.", "Gate A: Regression safety", "HUMAN_REVIEW")

    return ok

print("OK:", {"risk_log_initialized": True})

OK: {'risk_log_initialized': True}


##7.TRACE BUILDER

###7.1.OVERVIEW

**CELL 7 — TRACE BUILDER AND NORMALIZATION**

Cell 7 produces the reasoning_trace artifact, which is the audit ledger for the run. The trace is not a chain-of-thought transcript. It is a structured record of the pipeline’s behavior: inputs, boundaries, outputs, evaluations, and gates. This distinction is important for governance. We want transparency into process and evidence, not a dependency on internal model reasoning that is neither stable nor necessarily interpretable.

First, we normalize outputs into the contract shape. Normalization is defensive: it ensures that even if outputs are imperfect, the trace stores a stable representation for review. Where possible, missing fields are filled with safe defaults, and verification_status is forced to “Not verified.” This is not to hide policy failures; policy failures are logged in the risk log. The purpose is to keep the trace structurally consistent.

Then we build a trace that includes: run_id, timestamp, the pattern identifier (TRAINABLE_REASONING_EVAL_HARNESS), the input boundary, a hash of the synthetic case, hashes of prompts, normalized baseline and adapted outputs, and evaluation metrics. The trace also states which gates are enforced.

For a Board or auditor, the trace provides the answer to “what happened” in a compact, machine-readable way. It enables independent reconstruction without relying on notebook state. If this pattern is adopted in production, traces become the backbone of model risk management: they allow monitoring of drift, regression, and policy compliance over time.

Cell 7 therefore strengthens accountability. It ensures that the reasoning process is not just performed; it is recorded as evidence.

###7.2.CODE AND IMPLEMENTATION

In [9]:
# Cell 7 — Trace builder (reasoning_trace.json) + normalization
def normalize_ic_memo(obj: Any) -> Dict[str, Any]:
    if not isinstance(obj, dict):
        return {}
    # Ensure required keys exist (best-effort normalization; schema gate still authoritative)
    norm = {
        "executive_summary": str(obj.get("executive_summary", "")).strip(),
        "facts_provided": obj.get("facts_provided", {}) if isinstance(obj.get("facts_provided"), dict) else {},
        "assumptions_introduced": obj.get("assumptions_introduced", []) if isinstance(obj.get("assumptions_introduced"), list) else [],
        "open_items": obj.get("open_items", []) if isinstance(obj.get("open_items"), list) else [],
        "questions_to_verify": obj.get("questions_to_verify", []) if isinstance(obj.get("questions_to_verify"), list) else [],
        "analysis": str(obj.get("analysis", "")).strip(),
        "draft_output": str(obj.get("draft_output", "")).strip(),
        "verification_status": obj.get("verification_status", "Not verified"),
    }
    # Force disclaimer if missing (note: still log policy risk in gates)
    if norm["verification_status"] != "Not verified":
        norm["verification_status"] = "Not verified"
    return norm

def build_reasoning_trace(case: Dict[str, Any], baseline_raw: str, adapted_raw: str,
                          baseline_obj: Any, adapted_obj: Any,
                          quality_report: Dict[str, Any]) -> Dict[str, Any]:
    trace = {
        "run_id": CONFIG["run_id"],
        "timestamp_utc": now_utc_iso(),
        "pattern": "TRAINABLE_REASONING_EVAL_HARNESS",
        "input_boundary": case.get("input_boundary", {}),
        "case_hash": sha256_json(case),
        "prompts": {
            "baseline_raw_sha256": sha256_text(redact_text(baseline_raw)),
            "adapted_raw_sha256": sha256_text(redact_text(adapted_raw)),
        },
        "outputs": {
            "baseline": normalize_ic_memo(baseline_obj),
            "adapted": normalize_ic_memo(adapted_obj),
        },
        "evaluation": quality_report.get("metrics_comparison", {}),
        "gates": {
            "schema_and_policy": "enforced",
            "regression_safety": "enforced",
        },
        "notes": {
            "no_external_data": True,
            "verification_status_required": "Not verified",
        },
    }
    return trace

print("OK:", {"trace_ready": True})

OK: {'trace_ready': True}


##8.REPORT COMPOSER

###8.1.OVERVIEW

**CELL 8 — FINAL REPORT COMPOSER WITH STRICT SCHEMA**

Cell 8 converts the run outputs into a single board-facing final_report artifact. The final report is the official output of the pipeline: it contains facts_provided (the packet), assumptions_introduced, open_items, questions_to_verify, analysis, draft_output, verification_status, the full reasoning_quality_report, and the final recommendation with confidence.

The key governance move here is that the final report is also schema-validated. That means the pipeline’s “product” is contractually constrained. In finance, reporting formats matter because they determine what reviewers see and what can be checked systematically. By requiring a strict schema, we prevent silent omission of key governance fields.

The final report also enforces the “Not verified” stance. That is not a disclaimer for legal coverage; it is an operational control. It prevents the organization from treating model-generated content as verified fact. It keeps accountability where it belongs: with human diligence and sign-off.

If the final report schema fails, Cell 8 fails closed: it logs a risk, forces the recommendation to HUMAN_REVIEW, lowers confidence, and re-validates. If it still cannot validate, it raises an error rather than emitting an invalid report. This is consistent with control-grade posture: better to stop than to ship a broken artifact that could be misused.

In short, Cell 8 produces the artifact the Board actually reads, and it ensures that artifact is structurally and policy compliant.

###8.2.CODE AND IMPLEMENTATION

In [10]:
# Cell 8 — Report composer (final_report.json) with strict schema
def compose_final_report(case: Dict[str, Any],
                         quality_report: Dict[str, Any],
                         decision: str,
                         confidence: str) -> Dict[str, Any]:
    # Board-facing: keep it concise but control-grade; use adapted output as primary unless HUMAN_REVIEW
    adapted = quality_report["adapted_output"]
    baseline = quality_report["baseline_output"]
    chosen = adapted if decision == "PROMOTE_ADAPTED" else adapted  # still show adapted, but recommendation is HUMAN_REVIEW

    report = {
        "run_id": CONFIG["run_id"],
        "timestamp_utc": now_utc_iso(),
        "facts_provided": case,
        "assumptions_introduced": chosen.get("assumptions_introduced", []),
        "open_items": chosen.get("open_items", []),
        "questions_to_verify": chosen.get("questions_to_verify", []),
        "analysis": chosen.get("analysis", ""),
        "draft_output": chosen.get("draft_output", ""),
        "verification_status": "Not verified",
        "reasoning_quality_report": quality_report,
        "recommendation": decision,
        "confidence": confidence,
    }

    # Validate final report
    err = validate_or_none(FINAL_REPORT_SCHEMA, report)
    if err is not None:
        log_risk("high", "SCHEMA", f"Final report schema invalid: {err}", "Final report schema validation", "HUMAN_REVIEW")
        # Minimal hardening to meet schema (do not fabricate; just set safe defaults)
        report["recommendation"] = "HUMAN_REVIEW"
        report["confidence"] = "low"
        # Re-validate; if still invalid, raise (fail closed)
        err2 = validate_or_none(FINAL_REPORT_SCHEMA, report)
        if err2 is not None:
            raise RuntimeError(f"Final report cannot be validated even after hardening: {err2}")

    return report

print("OK:", {"final_report_schema_hash": sha256_json(FINAL_REPORT_SCHEMA)[:12]})

OK: {'final_report_schema_hash': 'd1e1ae1352bf'}


##9.EXECUTION

###9.1.0VERVIEW

**CELL 9 — END-TO-END ORCHESTRATOR: RUN, EVALUATE, LOG, DECIDE**

Cell 9 is the execution conductor. It ties together configuration, prompts, model calls, evaluation, gating, trace construction, risk logging, and final report writing.

First, it writes the run_manifest. The manifest captures run_id, timestamp, model, determinism settings, schema hashes, and a config hash. This is essential for reproducibility and for change management. It allows reviewers to answer: “What policy was in effect when this run was generated?” In production, manifests are the basis for audit trails and incident investigation.

Next, Cell 9 constructs the baseline and adapted prompts. The baseline prompt is comparatively generic. The adapted prompt explicitly includes a constraints library that forbids external facts and mandates open items and verification questions. Both prompts include the packet as the only source of truth.

Then Cell 9 runs two LLM calls. If either output is not valid JSON, the orchestrator logs a high-severity risk and replaces the output with a HUMAN_REVIEW-safe structure. This is a critical fail-closed behavior: it ensures the pipeline always produces reviewable artifacts, even on failure.

Cell 9 applies schema and policy gates to each output, generates the reasoning_quality_report via compare, validates that report against schema, and applies regression safety. The recommendation is PROMOTE_ADAPTED only if the adapted output passes critical gates and does not regress. Otherwise, the recommendation is HUMAN_REVIEW.

Finally, the orchestrator writes reasoning_trace.json, risk_log.json, and final_report.json. It returns a minimal console summary for operator visibility: decision, confidence, baseline and adapted overall scores, and risk count.

Cell 9 is therefore the operational core: a single command produces the entire governed run and its evidence bundle.

###9.2.CODE AND IMPLEMENTATION

In [11]:
# Cell 9 — Run orchestrator: executes pipeline end-to-end, writes artifacts
def build_prompts(case: Dict[str, Any]) -> Tuple[str, str]:
    packet_json = json.dumps(case, ensure_ascii=False, indent=2, sort_keys=True)

    baseline_user = (
        "TASK: Write a short investment-committee (IC) memo from the following packet.\n"
        "REQUIREMENTS:\n"
        "- Output JSON only.\n"
        "- Separate facts_provided, assumptions_introduced, open_items, questions_to_verify, analysis, draft_output.\n"
        "- verification_status must be 'Not verified'.\n"
        "- Use only facts in the packet.\n\n"
        f"PACKET:\n{packet_json}\n"
    )

    constraints_library = [
        "No external facts, market comps, industry stats, or sources.",
        "All numbers must come from packet or be clearly derived (and explain derivation).",
        "If a claim cannot be grounded, convert it into an open item or an assumption (labeled).",
        "Always include at least 5 open items and 5 questions_to_verify (use packet missing items).",
        "Board-ready: concise, structured, caveated, control-grade.",
    ]

    adapted_user = (
        "TASK: Produce a board-ready IC memo under GOVERNANCE CONTROLS.\n"
        "OUTPUT CONTRACT (JSON ONLY):\n"
        "Keys: executive_summary, facts_provided, assumptions_introduced, open_items, questions_to_verify, analysis, draft_output, verification_status.\n"
        "HARD CONTROLS:\n"
        f"- Constraints: {json.dumps(constraints_library, ensure_ascii=False)}\n"
        "- Allowed source: PACKET ONLY.\n"
        "- If you are tempted to add context (industry, macro, comps), DO NOT. Convert to open_items/questions.\n"
        "- verification_status MUST be 'Not verified'.\n\n"
        "PACKET (BOUND INPUT):\n"
        f"{packet_json}\n"
    )

    return baseline_user, adapted_user

def run_pipeline(case: Dict[str, Any]) -> Dict[str, Any]:
    # Run manifest
    run_manifest = {
        "run_id": CONFIG["run_id"],
        "timestamp_utc": now_utc_iso(),
        "model": CONFIG["model"],
        "determinism": {"PYTHONHASHSEED": os.environ.get("PYTHONHASHSEED"), "random_seed": 1337},
        "schemas": {
            "ic_memo_schema_sha256": sha256_json(IC_MEMO_SCHEMA),
            "quality_report_schema_sha256": sha256_json(QUALITY_REPORT_SCHEMA),
            "final_report_schema_sha256": sha256_json(FINAL_REPORT_SCHEMA),
        },
        "config_hash_sha256": sha256_json(CONFIG),
    }
    write_json(CONFIG["paths"]["run_manifest"], run_manifest)

    baseline_user, adapted_user = build_prompts(case)

    # LLM calls
    baseline_obj, baseline_raw = llm_json_call(
        prompt_name="baseline",
        system=SYSTEM_JSON_ONLY,
        user=baseline_user,
        max_tokens=CONFIG["max_tokens"],
        temperature=CONFIG["temperature"],
    )
    adapted_obj, adapted_raw = llm_json_call(
        prompt_name="adapted",
        system=SYSTEM_JSON_ONLY,
        user=adapted_user,
        max_tokens=CONFIG["max_tokens"],
        temperature=CONFIG["temperature"],
    )

    # Fail-closed if not parseable
    if baseline_obj is None:
        log_risk("high", "LLM_OUTPUT", "Baseline output not valid JSON.", "Strict JSON parse", "HUMAN_REVIEW")
        baseline_obj = {
            "executive_summary": "HUMAN_REVIEW: Baseline output was not valid JSON.",
            "facts_provided": case,
            "assumptions_introduced": [],
            "open_items": case.get("known_missing_items", []),
            "questions_to_verify": case.get("known_missing_items", []),
            "analysis": "No analysis: baseline JSON parse failed.",
            "draft_output": "HUMAN_REVIEW required.",
            "verification_status": "Not verified",
        }

    if adapted_obj is None:
        log_risk("high", "LLM_OUTPUT", "Adapted output not valid JSON.", "Strict JSON parse", "HUMAN_REVIEW")
        adapted_obj = {
            "executive_summary": "HUMAN_REVIEW: Adapted output was not valid JSON.",
            "facts_provided": case,
            "assumptions_introduced": [],
            "open_items": case.get("known_missing_items", []),
            "questions_to_verify": case.get("known_missing_items", []),
            "analysis": "No analysis: adapted JSON parse failed.",
            "draft_output": "HUMAN_REVIEW required.",
            "verification_status": "Not verified",
        }

    # Gates: schema + policy for both
    b_schema = gate_schema(baseline_obj, "baseline")
    a_schema = gate_schema(adapted_obj, "adapted")
    b_policy = gate_policy(baseline_obj, "baseline")
    a_policy = gate_policy(adapted_obj, "adapted")

    # Quality report
    quality_report = compare(baseline_obj, adapted_obj, case)

    # Gate B: schema enforcement on quality report
    q_err = validate_or_none(QUALITY_REPORT_SCHEMA, quality_report)
    if q_err is not None:
        log_risk("high", "SCHEMA", f"Quality report schema invalid: {q_err}", "Quality report schema validation", "HUMAN_REVIEW")
        # Fail closed: force HUMAN_REVIEW decision
        decision = "HUMAN_REVIEW"
        confidence = "low"
    else:
        # Regression gate
        bm = quality_report["metrics_comparison"]["baseline"]
        am = quality_report["metrics_comparison"]["adapted"]
        reg_ok = gate_regression_safety(bm, am)

        # Promote only if adapted passes key gates and regression safety
        if a_schema and a_policy and reg_ok:
            decision = "PROMOTE_ADAPTED"
            # Confidence heuristic: if adapted overall_score >= 0.85 and leak_rate low
            confidence = "high" if (am["overall_score"] >= 0.85 and am["assumption_leak_rate"] <= 0.05) else "medium"
        else:
            decision = "HUMAN_REVIEW"
            confidence = "low"

    # Reasoning trace
    trace = build_reasoning_trace(case, baseline_user, adapted_user, baseline_obj, adapted_obj, quality_report)
    write_json(CONFIG["paths"]["reasoning_trace"], trace)

    # Risk log + final report
    write_json(CONFIG["paths"]["risk_log"], {"run_id": CONFIG["run_id"], "timestamp_utc": now_utc_iso(), "risks": RISK_LOG})

    final_report = compose_final_report(case, quality_report, decision, confidence)
    write_json(CONFIG["paths"]["final_report"], final_report)

    return {
        "decision": decision,
        "confidence": confidence,
        "paths": CONFIG["paths"],
        "baseline_overall": quality_report["metrics_comparison"]["baseline"]["overall_score"],
        "adapted_overall": quality_report["metrics_comparison"]["adapted"]["overall_score"],
        "risk_count": len(RISK_LOG),
    }

RESULT = run_pipeline(CASE)
print("OK:", RESULT)

OK: {'decision': 'PROMOTE_ADAPTED', 'confidence': 'high', 'paths': {'run_manifest': '/content/artifacts/run_manifest.json', 'prompts_log': '/content/artifacts/prompts_log.jsonl', 'reasoning_trace': '/content/artifacts/reasoning_trace.json', 'risk_log': '/content/artifacts/risk_log.json', 'final_report': '/content/artifacts/final_report.json', 'deliverables_zip': '/content/deliverables/deliverables.zip'}, 'baseline_overall': 1.0, 'adapted_overall': 1.0, 'risk_count': 2}


##10.AUDIT BUNDLE

###10.1.OVERVIEW

**CELL 10 — PACKAGING AND OPERATOR SUMMARY**

Cell 10 packages the run outputs into a single zip file in deliverables/. This is a practical governance requirement. Evidence must be portable: it must be easy to archive, share with reviewers, or attach to internal approvals. A zip bundle is not glamorous, but it is the standard way to ensure nothing is missing.

The cell writes artifacts into the zip with consistent paths, typically under an artifacts/ directory, and includes convenience copies of key files at the root if desired. This helps reviewers who open the bundle quickly find the final_report.json.

Cell 10 then prints a minimal console summary verifying that required files exist. In institutional settings, operator feedback matters. Silent failures are dangerous because they create the illusion of compliance. The existence checks ensure that a run does not “look successful” while missing critical artifacts.

In the context of this notebook, Cell 10 also marks the boundary between analysis and operationalization. Everything before it is about reasoning and governance. This cell makes the outputs operationally usable: a self-contained deliverable that can be stored, transmitted, and reviewed without rerunning the notebook.

For Board relevance, this packaging step is a subtle but important signal: we are treating LLM outputs as governed deliverables, not ephemeral drafts. That posture is what enables consistent oversight and controlled adoption.

###10.2.CODE AND IMPLEMENTATION

####10.2.1.AUDIT BUNDLE

In [12]:
# Cell 10 — Packaging: zip deliverables + minimal console summary of outputs/paths
zip_path = CONFIG["paths"]["deliverables_zip"]
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in [
        CONFIG["paths"]["run_manifest"],
        CONFIG["paths"]["prompts_log"],
        CONFIG["paths"]["reasoning_trace"],
        CONFIG["paths"]["risk_log"],
        CONFIG["paths"]["final_report"],
    ]:
        if os.path.exists(p):
            z.write(p, arcname=os.path.join("artifacts", os.path.basename(p)))
    # Also include a copy of final_report at root for convenience
    if os.path.exists(CONFIG["paths"]["final_report"]):
        z.write(CONFIG["paths"]["final_report"], arcname="final_report.json")

print("DELIVERABLES_WRITTEN")
for k, v in CONFIG["paths"].items():
    if k != "deliverables_zip":
        print(k, "=>", v, "exists=", os.path.exists(v))
print("deliverables_zip =>", zip_path, "exists=", os.path.exists(zip_path))
print("SUMMARY =>", {"decision": RESULT["decision"], "confidence": RESULT["confidence"], "risk_count": RESULT["risk_count"]})

DELIVERABLES_WRITTEN
run_manifest => /content/artifacts/run_manifest.json exists= True
prompts_log => /content/artifacts/prompts_log.jsonl exists= True
reasoning_trace => /content/artifacts/reasoning_trace.json exists= True
risk_log => /content/artifacts/risk_log.json exists= True
final_report => /content/artifacts/final_report.json exists= True
deliverables_zip => /content/deliverables/deliverables.zip exists= True
SUMMARY => {'decision': 'PROMOTE_ADAPTED', 'confidence': 'high', 'risk_count': 2}


####10.2.2.REPORT

In [14]:
# Generate a board-ready prose report from the existing artifacts using Claude (packet-only, no invention).
# This cell reads artifacts/*, calls claude-haiku-4-5-20251001, and writes:
# - artifacts/board_report.txt
# - artifacts/board_report.json  (structured wrapper + the prose)
# - deliverables/deliverables.zip (updated to include the board report)

import os, json, re, hashlib, zipfile
import datetime
from typing import Any, Dict, Optional, Tuple, List

from google.colab import userdata

# --- UTC time (required pattern) ---
def now_utc_iso() -> str:
    return datetime.datetime.now(datetime.timezone.utc).isoformat()

# --- Paths ---
BASE_DIR = os.getcwd()
ARTIFACTS_DIR = os.path.join(BASE_DIR, "artifacts")
DELIVERABLES_DIR = os.path.join(BASE_DIR, "deliverables")
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
os.makedirs(DELIVERABLES_DIR, exist_ok=True)

PATHS = {
    "run_manifest": os.path.join(ARTIFACTS_DIR, "run_manifest.json"),
    "prompts_log": os.path.join(ARTIFACTS_DIR, "prompts_log.jsonl"),
    "reasoning_trace": os.path.join(ARTIFACTS_DIR, "reasoning_trace.json"),
    "risk_log": os.path.join(ARTIFACTS_DIR, "risk_log.json"),
    "final_report": os.path.join(ARTIFACTS_DIR, "final_report.json"),
    "board_report_txt": os.path.join(ARTIFACTS_DIR, "board_report.txt"),
    "board_report_json": os.path.join(ARTIFACTS_DIR, "board_report.json"),
    "deliverables_zip": os.path.join(DELIVERABLES_DIR, "deliverables.zip"),
}

# --- Helpers ---
def sha256_text(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

def sha256_json(obj: Any) -> str:
    return sha256_text(json.dumps(obj, sort_keys=True, separators=(",", ":")))

def write_json(path: str, obj: Any) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, sort_keys=True)

def append_jsonl(path: str, obj: Any) -> None:
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False, sort_keys=True) + "\n")

def read_json(path: str) -> Dict[str, Any]:
    if not os.path.exists(path):
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

PII_PATTERNS = [
    (re.compile(r"\b\d{3}-\d{2}-\d{4}\b"), "[REDACTED_SSN]"),
    (re.compile(r"\b(?:\+?\d{1,3}[\s-]?)?(?:\(?\d{2,3}\)?[\s-]?)?\d{3}[\s-]?\d{4}\b"), "[REDACTED_PHONE]"),
    (re.compile(r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b", re.IGNORECASE), "[REDACTED_EMAIL]"),
]

def redact_text(s: str, api_key: str) -> str:
    red = s
    for pat, repl in PII_PATTERNS:
        red = pat.sub(repl, red)
    if api_key:
        red = red.replace(api_key, "[REDACTED_KEY]")
    return red

# --- Load artifacts (source of truth) ---
run_manifest = read_json(PATHS["run_manifest"])
reasoning_trace = read_json(PATHS["reasoning_trace"])
risk_log = read_json(PATHS["risk_log"])
final_report = read_json(PATHS["final_report"])

missing = [k for k, p in PATHS.items() if k in ("run_manifest","reasoning_trace","risk_log","final_report") and not os.path.exists(p)]
if missing:
    raise FileNotFoundError(f"Missing required artifact files: {missing}. Ensure the notebook run produced them in /artifacts.")

# --- LLM client (Anthropic) ---
!pip -q install "anthropic>=0.49.0"
from anthropic import Anthropic

API_KEY = userdata.get("ANTHROPIC_API_KEY")
if not API_KEY:
    raise RuntimeError("Missing Colab Secret ANTHROPIC_API_KEY")

client = Anthropic(api_key=API_KEY)
MODEL = "claude-haiku-4-5-20251001"

def _extract_text(resp: Any) -> str:
    try:
        blocks = resp.content
        if isinstance(blocks, list) and blocks:
            parts = []
            for b in blocks:
                if hasattr(b, "text"):
                    parts.append(b.text)
            if parts:
                return "".join(parts)
    except Exception:
        pass
    return str(resp)

def log_prompt(prompt_name: str, system: str, user: str) -> None:
    entry = {
        "timestamp_utc": now_utc_iso(),
        "prompt_name": prompt_name,
        "model": MODEL,
        "system_sha256": sha256_text(system),
        "user_redacted_sha256": sha256_text(redact_text(user, API_KEY)),
        "user_redacted_preview": redact_text(user, API_KEY)[:700],
    }
    append_jsonl(PATHS["prompts_log"], entry)

# --- Build the board report prompt (artifact-grounded, no invention) ---
SYSTEM = (
    "You are a governance-first finance analyst writing to a Board of Directors. "
    "You MUST ground every concrete detail in the provided JSON artifacts. "
    "Do NOT invent numbers, outcomes, or facts. If something is missing, explicitly say it is not present in the artifacts. "
    "Write in polished, board-ready prose with clear structure, but remain strictly faithful to the artifacts. "
    "The report MUST state verification_status = 'Not verified' and explain what that means. "
    "Output must be PLAIN TEXT only (no markdown fences, no JSON)."
)

# Keep payload bounded but complete enough:
payload = {
    "run_manifest": run_manifest,
    "reasoning_trace": reasoning_trace,
    "risk_log": risk_log,
    "final_report": final_report,
}

USER = (
    "Task: Write a rich, well-explained Board report describing the results of the Trainable Reasoning evaluation run.\n"
    "You are given four artifacts as JSON:\n"
    "1) run_manifest (run configuration + schema hashes + determinism)\n"
    "2) reasoning_trace (process trace, evaluation metrics, and notes)\n"
    "3) risk_log (all risks logged with controls and statuses)\n"
    "4) final_report (decision recommendation + confidence + reasoning_quality_report)\n\n"
    "Required sections (use clear headings):\n"
    "A. Executive Summary (1 page equivalent)\n"
    "B. What We Ran and Why (baseline vs adapted; measured improvement; governance-first framing)\n"
    "C. Inputs and Boundaries (packet-only rule; what is disallowed; determinism)\n"
    "D. Process Walkthrough (end-to-end): prompt modes, JSON contract, evaluation harness, trace production\n"
    "E. The Reasoning Trace (what it records; what it proves; what it does not prove)\n"
    "F. Gates and Controls (schema gate, policy checks, regression safety; include which gates passed/failed and evidence)\n"
    "G. Results (baseline vs adapted metrics: assumption_leak_rate, uncertainty_disclosure_rate, schema_validity, policy_violations, overall_score)\n"
    "H. Risk Log Summary (top risks, severity distribution, and what triggered HUMAN_REVIEW if applicable)\n"
    "I. Final Recommendation (PROMOTE_ADAPTED or HUMAN_REVIEW) + confidence rationale strictly from artifacts\n"
    "J. Operational Constraints for Production (what controls must remain; what humans must review)\n"
    "K. Limitations and Next Steps (no external verification; suggested hardening steps)\n\n"
    "Hard constraints:\n"
    "- Do not introduce any market data, sources, or facts not in the artifacts.\n"
    "- If a figure is not present, say so.\n"
    "- Keep tone board-ready and control-grade.\n"
    "- Include 'Verification Status: Not verified' and explain it.\n\n"
    "Here are the artifacts JSON (source of truth):\n"
    f"{json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True)}\n"
)

log_prompt("board_report_generation", SYSTEM, USER)

resp = client.messages.create(
    model=MODEL,
    max_tokens=1800,
    temperature=0.2,  # small amount for prose fluency; still artifact-grounded by instruction
    system=SYSTEM,
    messages=[{"role": "user", "content": USER}],
)

board_text = _extract_text(resp).strip()

# --- Write outputs ---
with open(PATHS["board_report_txt"], "w", encoding="utf-8") as f:
    f.write(board_text + "\n")

board_wrapper = {
    "run_id": final_report.get("run_id") or run_manifest.get("run_id") or "unknown",
    "timestamp_utc": now_utc_iso(),
    "verification_status": "Not verified",
    "board_report_text_sha256": sha256_text(board_text),
    "board_report_text_path": PATHS["board_report_txt"],
    "source_artifacts": {
        "run_manifest_path": PATHS["run_manifest"],
        "reasoning_trace_path": PATHS["reasoning_trace"],
        "risk_log_path": PATHS["risk_log"],
        "final_report_path": PATHS["final_report"],
    },
}
write_json(PATHS["board_report_json"], board_wrapper)

# --- Update deliverables zip to include board report ---
zip_path = PATHS["deliverables_zip"]
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in [
        PATHS["run_manifest"],
        PATHS["prompts_log"],
        PATHS["reasoning_trace"],
        PATHS["risk_log"],
        PATHS["final_report"],
        PATHS["board_report_txt"],
        PATHS["board_report_json"],
    ]:
        if os.path.exists(p):
            z.write(p, arcname=os.path.join("artifacts", os.path.basename(p)))
    # Convenience copies at root
    if os.path.exists(PATHS["final_report"]):
        z.write(PATHS["final_report"], arcname="final_report.json")
    if os.path.exists(PATHS["board_report_txt"]):
        z.write(PATHS["board_report_txt"], arcname="board_report.txt")

print("BOARD REPORT WRITTEN")
print(" -", PATHS["board_report_txt"])
print(" -", PATHS["board_report_json"])
print("DELIVERABLES ZIP UPDATED")
print(" -", zip_path)
print("\n--- BOARD REPORT PREVIEW (first 1200 chars) ---\n")
print(board_text[:4200])

BOARD REPORT WRITTEN
 - /content/artifacts/board_report.txt
 - /content/artifacts/board_report.json
DELIVERABLES ZIP UPDATED
 - /content/deliverables/deliverables.zip

--- BOARD REPORT PREVIEW (first 1200 chars) ---

TRAINABLE REASONING EVALUATION RUN — BOARD REPORT

Run ID: 08f03fcfd0676eab
Timestamp: 2026-02-21T12:26:31 UTC
Model: claude-haiku-4-5-20251001

Verification Status: Not verified

This evaluation run produced outputs that failed JSON schema validation in both baseline and adapted modes. The reasoning trace and risk log document these failures. No external verification of the underlying reasoning quality has been performed. All metrics and findings reported below are derived solely from the artifacts provided and reflect the state of the evaluation harness at execution time.

---

A. EXECUTIVE SUMMARY

This evaluation compared a baseline prompt configuration against an adapted variant, both tasked with drafting a board-ready investment committee memo for Northbridge Compone

##11.CONCLUSION

**CONCLUSION AND BRIDGE FORWARD (BOARD-FACING)**

This notebook makes a specific, positive contribution to the organization’s AI program: it demonstrates that “trainable reasoning” can be implemented as a controlled, measurable workflow rather than an aspirational slogan. In practical governance terms, we moved from “prompting as an art” to “prompting as a change-managed control surface.” The baseline-versus-adapted experiment is not interesting because the model wrote different prose. It is interesting because we can now test and promote a configuration only when it clears explicit safety gates, and we can retain evidence of why that decision was made.

The first positive contribution is the transformation of LLM usage into an auditable pipeline with standard artifacts. Every run generates a manifest, a prompts ledger, a reasoning trace, a risk register, and a final board-facing report. This is the minimum viable infrastructure for institutional accountability. A controlled process must be reconstructible after the fact; otherwise it is merely a workflow convenience. The governance bundle means that if an output is questioned—internally, by auditors, or by regulators—we can show the exact configuration and the exact control outcomes that produced it.

The second positive contribution is strict separation of facts, assumptions, and open items. This is not stylistic hygiene; it is a defense against the most common failure mode of generative models in finance: presenting plausible narrative as if it were grounded evidence. By forcing an explicit facts_provided field and a mandatory “Not verified” disclaimer, the notebook encodes a professional posture: outputs are drafts requiring diligence, not assertions of truth. By requiring open items and verification questions, the pipeline resists the organizational tendency to treat fluent text as complete analysis.

The third positive contribution is deterministic evaluation. The rubric—assumption leak rate, uncertainty disclosure rate, schema validity, and policy violations—is applied by code, not by the model. That distinction matters. If the model both generates and judges, the system loses independence. Here, the evaluation harness is the control layer: it is consistent run-to-run, it is explainable, and it produces measurable evidence. This is the bridge from “LLM outputs” to “governed outputs.”

The fourth positive contribution is regression safety as a gate, not a suggestion. When we propose an adapted prompt (a stricter template), we do not assume it is safer. We test it and we require that it does not regress on critical risk metrics. If it regresses, the system fails closed into HUMAN_REVIEW, and the risk log records the specific failures. This is precisely how mature financial institutions manage model and policy change: a change is not an improvement unless it can be evidenced as such.

At the same time, the notebook has clear limitations, and the Board should understand them as design boundaries rather than shortcomings.

First, the system does not verify truth. It does not connect to external data sources, and it does not prove that the memo’s narrative is correct. This is intentional: verification and data provenance are separate architectural layers that require connectors, permissions, and audit controls. Today’s notebook demonstrates the reasoning governance mechanism in a synthetic environment. It prepares the ground for real data integration, but it does not perform it.

Second, the assumption leak metric is a heuristic. It is conservative and useful, but it is not a full semantic grounding solution. It primarily detects numeric hallucinations and some contract breaches. Future work should extend grounding controls toward claim-level mapping: for each claim in analysis, the system should cite the input key(s) supporting it, or else route it into assumptions/open items. That is feasible but requires additional parsing, structured claim extraction, and stricter evaluation.

Third, the scoring model is a policy choice. We used explicit weights and thresholds to make the demonstration clear. In production, those weights must be approved by governance owners (risk, compliance, and the business line), and the thresholds should be calibrated on representative internal drafting tasks. The important point is not the exact weights; it is the existence of a formal scoring policy with documented rationale and change management.

Fourth, the notebook evaluates prompt changes, not organizational outcomes. A safer prompt does not automatically translate into safer decisions unless humans follow the workflow: read the open items, verify the claims, and enforce sign-off. Governance is socio-technical. This notebook produces controls and evidence; operational adoption must include training, accountability, and clear usage policies.

So what did we achieve? We demonstrated that we can operationalize “trainable reasoning” as a controlled improvement cycle: propose constraints, run baseline vs adapted, score deterministically, apply regression gates, produce evidence bundles, and either promote or escalate. That is a meaningful step toward enterprise-grade deployment.

What can it not do yet? It cannot ground narrative claims with full semantic rigor; it cannot ingest and validate external data with provenance; it cannot run quantitative stress tests or scenario analytics; and it cannot replace human fiduciary judgment. In its current form, it is a drafting and governance mechanism—not a decision engine.

This sets a clean bridge to future chapters.

The next layer is richer grounding and traceability: extending the trace so each claim is explicitly linked to a source key or flagged as an assumption. That is “contract-level grounding” and is a natural continuation of this notebook’s schema-first approach.

The following layer is integration with deterministic quantitative analytics: stress testing, sensitivity analysis, concentration measures, and scenario tables computed outside the model. That will shift more of the analysis from narrative to auditable computation.

Beyond that, we move into institutional workflow architecture: committee reasoning, sign-off gates, and immutable artifact registries. The Board should expect the system to evolve into an end-to-end governed pipeline where human approvals are first-class events, not afterthoughts, and where artifacts are archived with versioned policies and controlled access.

In other words: this notebook provides the “quality improvement harness” for reasoning. Future chapters will add (i) deeper grounding, (ii) more deterministic analytics, and (iii) enterprise workflow integration—while preserving the same principle that defines this program: capability increases must be matched by stronger controls and better evidence.